In [8]:
import os
import json
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time
import random
import re
from datetime import datetime
import logging
from typing import List, Dict, Optional
from transformers import AutoTokenizer, AutoModel  # 引入模型加载库
import torch

In [9]:
# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("crawler.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

In [32]:
class KeywordExtractor:
    """基于ChatGLM-6B的金融领域关键词提取器（无需jieba）"""
    
    def __init__(self, model_path: str = "THUDM/chatglm-6b-int4", device: str = "cpu"):
        """
        初始化大模型关键词提取器
        
        Args:
            model_path: 模型路径（Hugging Face模型名或本地路径）
            device: 运行设备（'cpu'或'cuda'）
        """
        self.device = device
        self.model_path = model_path
        logger.info(f"初始化关键词提取器，模型: {model_path}，设备: {device}")
        
        # 加载模型（INT4量化，低内存占用）
        self.tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        self.model = AutoModel.from_pretrained(model_path, trust_remote_code=True).quantize(4).to(device)
        self.model.eval()
        logger.info("模型加载完成")
        
        # 优化的提示词模板（金融领域专用）
        self.prompt_template = """
        你是专业金融关键词提取引擎，请按以下规则从问题中提取关键词：
        1. 提取上市公司名称（如"贵州茅台"）、行业（如"白酒"）、金融产品（如"沪深300"）
        2. 标准化时间（如"今年"→"2024年"，"Q1"→"第一季度"）
        3. 保留完整财务指标（如"营业收入"、"净利润率"）
        4. 去除无意义词（如"如何"、"请问"）
        5. 合并同类词（如"股价"→"股票价格"）
        
        输出格式：英文逗号分隔的关键词列表，示例：
        问题："2024年央行降准对A股的影响"
        输出：2024年,央行降准,A股市场,影响
        
        问题：{question}
        输出：
        """
    
    def extract(self, question: str, max_retries: int = 3) -> List[str]:
        """
        从问题中提取金融关键词（含重试机制）
        
        Args:
            question: 输入问题（如"贵州茅台2024年营收预测"）
            max_retries: 模型调用失败时的最大重试次数
            
        Returns:
            标准化关键词列表（如["贵州茅台", "2024年", "营业收入", "预测"]）
        """
        for retry in range(max_retries):
            try:
                # 1. 调用大模型提取关键词
                keywords = self._extract_with_llm(question)
                # 2. 后处理标准化
                keywords = self._post_process(keywords)
                if keywords:
                    logger.info(f"提取成功，关键词: {keywords}")
                    return keywords
                
                # 3. 模型失败时启用正则备用
                logger.warning(f"模型提取失败，重试 {retry+1}/{max_retries}")
                keywords = self._extract_with_regex(question)
                if keywords:
                    logger.info(f"正则提取成功，关键词: {keywords}")
                    return keywords
                
            except Exception as e:
                logger.error(f"提取失败 ({retry+1}/{max_retries}): {e}")
                if retry == max_retries - 1:
                    logger.error(f"最终失败，返回空列表，问题: {question}")
                    return []
            time.sleep(1)  # 重试间隔1秒
    
    def _extract_with_llm(self, question: str) -> List[str]:
        """使用ChatGLM提取关键词（含提示词优化）"""
        prompt = self.prompt_template.format(question=question)
        response, _ = self.model.chat(
            self.tokenizer,
            prompt,
            max_length=256,
            temperature=0.1,  # 低温度保证输出确定性
            stop=["\n", "。", "；", "，"]  # 强制终止符
        )
        # 解析输出（支持多种分隔符）
        raw = response.strip()
        separators = ["，", "、", ","]
        for sep in separators:
            if sep in raw:
                return [kw.strip() for kw in raw.split(sep) if kw.strip()]
        return [raw] if raw else []
    
    def _post_process(self, keywords: List[str]) -> List[str]:
        """关键词后处理（标准化、去重、过滤）"""
        if not keywords:
            return []
        
        # 1. 过滤停用词
        stopwords = {"如何", "什么", "为什么", "请问", "吗", "呢", "的", "了", "是", "在"}
        keywords = [kw for kw in keywords if kw not in stopwords and len(kw) >= 2]
        
        # 2. 时间标准化（示例映射）
        time_map = {"今年": "2024年", "去年": "2023年", "Q1": "第一季度", "Q2": "第二季度"}
        keywords = [time_map.get(kw, kw) for kw in keywords]
        
        # 3. 同义词合并（示例映射）
        synonym_map = {"营收": "营业收入", "净利": "净利润", "GDP": "国内生产总值"}
        keywords = [synonym_map.get(kw, kw) for kw in keywords]
        
        # 4. 去重（保持顺序）
        seen = set()
        return [kw for kw in keywords if not (kw in seen or seen.add(kw))]
    
    def _extract_with_regex(self, question: str) -> List[str]:
        """正则表达式备用提取（模型失效时）"""
        patterns = [
            r'[A-Za-z0-9\u4e00-\u9fa5]{2,10}公司',       # 公司名
            r'[0-9]{4}年(一季度|二季度|三季度|四季度|\d{1,2}月)',  # 时间
            r'[营收|利润|GDP|CPI|利率|汇率|市盈率|市值]',  # 财务指标
            r'[A-Z]{2,6}指数',  # 指数名
            r'[沪深|上证|深证|创业板|科创板]市场?',  # 市场板块
        ]
        keywords = []
        for pattern in patterns:
            matches = re.findall(pattern, question)
            keywords.extend(matches)
        return list(set([kw for kw in keywords if len(kw) >= 2]))

In [11]:
class DataSource:
    """数据源抽象类"""
    
    def __init__(self, name: str, base_url: str):
        self.name = name
        self.base_url = base_url
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
    
    def search(self, keywords: List[str]) -> List[Dict]:
        """根据关键词搜索数据（需子类实现）"""
        raise NotImplementedError
    
    def parse(self, response: requests.Response) -> List[Dict]:
        """解析响应数据（需子类实现）"""
        raise NotImplementedError

In [40]:
class StatsDataSource:
    """统计数据爬取器（基于国家统计局搜索接口）"""
    
    def __init__(self):
        """初始化爬取器，设置搜索数据源"""
        self.data_cache = {}  # 数据缓存
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
            "Referer": "https://www.stats.gov.cn/"
        }
        logger.info("统计数据爬取器初始化完成")
        
        # 基于搜索的数据源（最新URL）
        self.search_datasources = {
            "gdp": "https://www.stats.gov.cn/search/s?qt=gdp",
            "cpi": "https://www.stats.gov.cn/search/s?qt=cpi",
            "ppi": "https://www.stats.gov.cn/search/s?qt=ppi",
            "export": "https://www.stats.gov.cn/search/s?qt=出口",
            "interest": "https://www.stats.gov.cn/search/s?qt=利率",
        }
    
    def fetch_macro_data(self, indicator: str, years: int = 5) -> pd.DataFrame:
        """
        通过搜索接口获取宏观经济数据
        
        Args:
            indicator: 指标类型（"gdp", "cpi"等）
            years: 获取最近years年的数据
            
        Returns:
            结构化DataFrame（含日期、数值列）
        """
        if indicator not in self.search_datasources:
            logger.error(f"不支持的指标: {indicator}，可选: {list(self.search_datasources.keys())}")
            return pd.DataFrame()
        
        # 从缓存获取数据
        if indicator in self.data_cache:
            logger.info(f"使用缓存的{indicator}数据")
            return self.data_cache[indicator].copy()
        
        # 从搜索接口爬取数据
        df = self._crawl_search_data(indicator)
        if not df.empty:
            self.data_cache[indicator] = df
            logger.info(f"成功爬取{indicator}数据，记录数: {len(df)}")
        
        # 筛选最近years年数据
        if not df.empty and "date" in df.columns:
            latest_year = int(df["date"].iloc[0].split("-")[0])
            start_year = latest_year - years
            df = df[df["date"].str.startswith(str(start_year))]
            logger.info(f"筛选{indicator}数据，时间范围: {start_year}-{latest_year}")
        
        return df
    
    # -------------- 数据获取私有方法 --------------
    def _crawl_search_data(self, indicator: str) -> pd.DataFrame:
        """爬取搜索结果页面数据"""
        url = self.search_datasources[indicator]
        try:
            # 增加随机延时防爬
            import time, random
            time.sleep(random.uniform(1, 3))
            
            response = requests.get(url, headers=self.headers, timeout=20)
            response.raise_for_status()
            html = response.text
            return self._parse_search_html(html, indicator)
            
        except Exception as e:
            logger.error(f"爬取{indicator}数据失败: {e}，URL: {url}")
            return pd.DataFrame()
    
    def _parse_search_html(self, html: str, indicator: str) -> pd.DataFrame:
        """解析搜索结果页面（以GDP为例）"""
        soup = BeautifulSoup(html, 'html.parser')
        results = []
        
        #解析GDP搜索结果
        if indicator == "gdp":
            # 查找包含GDP数据的表格
            tables = soup.select(".public-table") or soup.select(".conMidtab table")
            for table in tables:
                try:
                    rows = table.select("tr")
                    for row in rows[1:]:  # 跳过表头
                        cols = row.select("td")
                        if len(cols) >= 2:
                            date = cols[0].text.strip()
                            value = cols[1].text.strip() if len(cols) >= 2 else "N/A"
                            
                            # 数值转换（示例）
                            if "%" in value:
                                value = float(value.replace("%", ""))
                            elif "%" not in value and value.replace(".", "", 1).isdigit():
                                value = float(value)
                            
                            results.append({"date": date, "value": value})
                except:
                    continue
        
        return pd.DataFrame(results)

In [30]:
class StockExchangeDataSource(DataSource):
    """证券交易所数据源（支持多交易所，简化版）"""
    
    # 交易所配置：代码 -> (名称, 搜索URL模板)
    timestamp = int(time.time() * 1000)
    EXCHANGES = {
        "sse": ("上海证券交易所", "https://www.sse.com.cn/home/search/?webswd={keywords}"),
        "szse": ("深圳证券交易所", "https://www.szse.cn/application/search/index.html?keyword={keywords}&r={timestamp}"),
    }
    
    def __init__(self, exchange_code: str):
        """
        Args:
            exchange_code: 交易所代码，如 "sse"（上交所）、"szse"（深交所）
        """
        if exchange_code not in self.EXCHANGES:
            raise ValueError(f"不支持的交易所: {exchange_code}，可选: {list(self.EXCHANGES.keys())}")
        
        name, search_url = self.EXCHANGES[exchange_code]
        super().__init__(name, search_url)
        self.exchange_code = exchange_code
    
    def search(self, keywords: List[str]) -> List[Dict]:
        """搜索公告信息"""
        if not keywords:
            logger.warning("搜索关键词为空")
            return []
        
        # 拼接关键词（用+连接）
        keyword_str = '+'.join(keywords)
        logger.info(f"在{self.name}搜索: {keyword_str}")
        
        # 生成搜索URL（替换{keywords}为实际关键词）
        search_url = self.base_url.replace("{keywords}", keyword_str)
        
        # 发起请求并解析
        results = self._fetch_and_parse(search_url)
        
        # 确保默认结果包含exchange字段
        if not results:
            results = [{"title": f"{self.name}公告: {keyword_str}", 
                       "url": search_url,
                       "exchange": self.name}]
        
        return results
    
    def _fetch_and_parse(self, url: str) -> List[Dict]:
        """获取网页并解析结果"""
        try:
            # 随机延时（防爬）
            time.sleep(random.uniform(1, 3))
            
            # 发起请求
            headers = {
                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
                "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8"
            }
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()  # 检查请求是否成功
            
            # 解析页面（通用逻辑，实际需根据各交易所页面结构调整）
            return self._parse_html(response.text)
            
        except Exception as e:
            logger.error(f"搜索失败: {e}")
            return []
    
    def _parse_html(self, html: str) -> List[Dict]:
        """解析HTML页面获取公告信息（需根据实际页面调整）"""
        soup = BeautifulSoup(html, 'html.parser')
        results = []
        
        # 解析逻辑
        if self.exchange_code in ["sse", "szse"]:
            # 解析上交所/深交所公告
            if self.exchange_code == "sse":
                items = soup.select(".result-list__item")
            else:  # szse
                items = soup.select(".list-content-item")
                
            for item in items[:5]:  # 取前5条结果
                try:
                    title = item.select_one(".title").text.strip() if item.select_one(".title") else "未知标题"
                    url = item.select_one("a")["href"] if item.select_one("a") else "#"
                    date = item.select_one(".date").text.strip() if item.select_one(".date") else "未知日期"
                    results.append({
                        "title": title,
                        "url": url if url.startswith("http") else self.base_url + url,
                        "date": date,
                        "exchange": self.name
                    })
                except:
                    continue
        elif self.exchange_code == "hkex":
            # 解析港交所公告
            items = soup.select(".news-item")
            for item in items[:5]:
                try:
                    title = item.select_one(".news-title").text.strip()
                    url = item.select_one("a")["href"]
                    results.append({
                        "title": title,
                        "url": url,
                        "exchange": self.name
                    })
                except:
                    continue
        elif self.exchange_code == "nyse":
            # 解析纽交所公告
            items = soup.select(".search-result-item")
            for item in items[:5]:
                try:
                    title = item.select_one(".title").text.strip()
                    url = item.select_one("a")["href"]
                    results.append({
                        "title": title,
                        "url": url,
                        "exchange": self.name
                    })
                except:
                    continue
        
        return results


In [15]:
class FinancialCrawler:
    """金融研报爬虫主类"""
    
    def __init__(self, model_path="THUDM/chatglm-6b-int4"):
        self.keyword_extractor = KeywordExtractor(model_path)
        self.data_sources = [
            WindDataSource(),
            StockExchangeDataSource(),
            IndustryReportDataSource()
        ]
        self.results_dir = "report_results"
        os.makedirs(self.results_dir, exist_ok=True)
    
    def read_questions_from_file(self, file_path: str) -> List[str]:
        """从文件读取问题"""
        if file_path.endswith('.txt'):
            with open(file_path, 'r', encoding='utf-8') as f:
                return [line.strip() for line in f if line.strip()]
        elif file_path.endswith(('.csv', '.xlsx', '.xls')):
            df = pd.read_csv(file_path) if file_path.endswith('.csv') else pd.read_excel(file_path)
            return df.iloc[:, 0].tolist() if not df.empty else []
        return []
    
    def crawl_report(self, question: str, max_results: int = 3) -> Dict:
        """爬取单个问题的研报数据"""
        keywords = self.keyword_extractor.extract_keywords(question)
        if not keywords:
            logger.warning(f"未提取到关键词: {question}")
            return {"question": question, "keywords": [], "reports": []}
        
        reports = []
        for source in self.data_sources:
            try:
                source_reports = source.search(keywords)[:max_results]
                reports.extend(source_reports)
                logger.info(f"从{source.name}获取{len(source_reports)}条研报")
            except Exception as e:
                logger.error(f"{source.name}爬取失败: {e}")
        
        return {
            "question": question,
            "keywords": keywords,
            "reports": reports
        }
    
    def batch_crawl(self, questions: List[str], output_file: str = None) -> List[Dict]:
        """批量爬取多个问题的研报"""
        all_results = []
        for i, question in enumerate(questions):
            logger.info(f"处理问题 {i+1}/{len(questions)}: {question}")
            result = self.crawl_report(question)
            all_results.append(result)
            
            # 定期保存结果
            if (i+1) % 5 == 0 or (i+1) == len(questions):
                if output_file:
                    self.save_results(all_results, output_file)
                logger.info(f"已完成 {i+1} 个问题")
            
            # 随机延迟避免反爬
            time.sleep(random.uniform(2, 5))
        
        return all_results
    
    def save_results(self, results: List[Dict], output_file: str) -> None:
        """保存爬取结果到JSON文件"""
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        logger.info(f"结果已保存至 {output_file}")


In [16]:
# 使用示例
if __name__ == "__main__":
    # 创建爬虫实例（可指定本地模型路径）
    crawler = FinancialCrawler(model_path="THUDM/chatglm-6b-int4")
    
    # 示例问题
    sample_questions = [
        "2023年宁德时代营收情况",
        "半导体行业2024年投资趋势",
        "中国GDP增长对股市影响分析"
    ]
    
    # 批量爬取研报
    results = crawler.batch_crawl(sample_questions, output_file="report_results.json")
    
    # 打印结果摘要
    for result in results:
        print(f"\n问题: {result['question']}")
        print(f"关键词: {', '.join(result['keywords'])}")
        print(f"找到 {len(result['reports'])} 篇研报:")
        for i, report in enumerate(result['reports']):
            print(f"  {i+1}. {report['title']} - {report['url']}")

2025-06-16 15:58:20,763 - __main__ - INFO - 正在加载模型: THUDM/chatglm-6b-int4，设备: cpu
D:\Users\27526\Anaconda3\envs\open_source_llm\lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


OSError: We couldn't connect to 'https://huggingface.co' to load this file, couldn't find it in the cached files and it looks like THUDM/chatglm-6b-int4 is not the path to a directory containing a file named config.json.
Checkout your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.